In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain_experimental.agents import create_pandas_dataframe_agent
import pandas as pd

def create_sample_data(path: str):
    """Creates a sample sales dataset for demo."""
    import random
    from datetime import date, timedelta

    random.seed(42)
    rows = []
    products = ["Laptop", "Phone", "Tablet", "Monitor", "Keyboard"]
    regions = ["North", "South", "East", "West"]
    start = date(2024, 1, 1)

    for i in range(200):
        d = start + timedelta(days=random.randint(0, 364))
        rows.append({
            "date": d.isoformat(),
            "product": random.choice(products),
            "region": random.choice(regions),
            "quantity": random.randint(1, 20),
            "unit_price": round(random.uniform(50, 2000), 2),
            "revenue": 0,
        })

    df = pd.DataFrame(rows)
    df["revenue"] = df["quantity"] * df["unit_price"]
    df.to_csv(path, index=False)
    return df

/var/folders/hl/xhb5c58j7ydchwx0hwc5nzcr0000gn/T/ipykernel_35451/1966044361.py:1: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.agents import create_pandas_dataframe_agent


In [4]:
df = create_sample_data("/Users/linqibin/Documents/code/agents/practise_agents/apps/backend/documents/csv/sample_data.csv")

In [5]:
import os

model = os.getenv('MODEL_NAME', '')

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model=model, temperature=0.3, extra_body={'enabel_thinking': False})

agent = create_pandas_dataframe_agent(llm=llm, df=df, verbose=True, allow_dangerous_code=True, agent_type="openai-tools")

In [8]:
for chunk in agent.stream(input="每个月的销售额趋势是怎样的？"):
    print(chunk)

# agent.invoke('每个月的销售额趋势是怎样的？')



> Entering new None chain...
{'actions': [ToolAgentAction(tool='python_repl_ast', tool_input={'query': "import pandas as pd\nimport matplotlib.pyplot as plt\n\n# 确保 date 列是 datetime 类型\ndf['date'] = pd.to_datetime(df['date'])\n\n# 提取年份和月份作为索引\ndf['year_month'] = df['date'].dt.to_period('M')\n\n# 按月份分组并计算总销售额\nmonthly_revenue = df.groupby('year_month')['revenue'].sum().reset_index()\nmonthly_revenue['year_month'] = monthly_revenue['year_month'].astype(str)\n\n# 按时间排序\nmonthly_revenue = monthly_revenue.sort_values('year_month')\n\n# 绘制趋势图\nplt.figure(figsize=(12, 6))\nplt.plot(monthly_revenue['year_month'], monthly_revenue['revenue'], marker='o', linestyle='-')\nplt.title('Monthly Sales Revenue Trend')\nplt.xlabel('Month')\nplt.ylabel('Revenue')\nplt.xticks(rotation=45)\nplt.grid(True)\nplt.tight_layout()\nplt.show()\n\n# 打印数据\nprint(monthly_revenue)"}, log='\nInvoking: `python_repl_ast` with `{\'query\': "import pandas as pd\\nimport matplotlib.pyplot as plt\\n\\n# 确保 date 列是 datetime

In [ ]:
# Stream the final answer token by token.
question = "每个月的销售额趋势是怎样的？"
is_answer_started = False

agent = create_pandas_dataframe_agent(llm=llm, df=df, verbose=False, allow_dangerous_code=True, agent_type="openai-tools")

async for event in agent.astream_events({"input": question}, version="v2"):
    if event["event"] == "on_tool_start":
        print(f"\n调用工具: {event['name']}")
    elif event["event"] == "on_tool_end":
        print("工具调用完成，正在生成回答...")
    elif event["event"] == "on_chat_model_stream":
        if 'chunk' in event['data']:
            token = event["data"]["chunk"].content
            if isinstance(token, str) and token:
                if not is_answer_started:
                    print("\n最终回答:")
                    is_answer_started = True
                print(token, end="", flush=True)





调用工具: python_repl_ast
工具调用完成，正在生成回答...

最终回答:
根据数据，2024年各月的销售额（revenue）趋势如下：

| 月份 | 销售额 |
|------|--------|
| 2024-01 | 188,879.24 |
| 2024-02 | 212,894.75 |
| 2024-03 | 142,107.91 |
| 2024-04 | 128,899.83 |
| 2024-05 | 277,043.09 |
| 2024-06 | 133,285.90 |
| 2024-07 | 172,698.66 |
| 2024-08 | 199,413.54 |
| 2024-09 | 191,502.93 |
| 2024-10 | 199,078.42 |
| 2024-11 | 150,621.15 |
| 2024-12 | 223,394.64 |

**趋势分析：**

1. **整体波动较大**：销售额在12.9万到27.7万之间波动，没有呈现单一的上升或下降趋势。

2. **高峰期**：
   - **5月**达到全年最高峰（277,043.09）
   - **12月**次高（223,394.64）
   - **2月**也较高（212,894.75）

3. **低谷期**：
   - **4月**为全年最低（128,899.83）
   - **6月**次低（133,285.90）
   - **3月**和**11月**也相对较低

4. **季度表现**：
   - Q1（1-3月）：先升后降
   - Q2（4-6月）：波动剧烈，4月最低，5月最高
   - Q3（7-9月）：相对稳定，维持在17-20万之间
   - Q4（10-12月）：11月下滑，12月大幅反弹

总体而言，销售额呈现**周期性波动**特征，可能在某些月份有促销活动或季节性因素影响。